In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OOM-Reproduction") \
    .master("local[*]") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Spark version: 3.5.0
Master: local[*]
Default parallelism: 12


In [2]:
df = spark.createDataFrame(
    [(1, "Jaya"), (2, "Test")],
    ["id", "name"]
)

df.show()

+---+----+
| id|name|
+---+----+
|  1|Jaya|
|  2|Test|
+---+----+



In [3]:
df = spark.range(0, 50_000_000)

print("Partitions:", df.rdd.getNumPartitions())
print("Records:", df.count())

Partitions: 12
Records: 50000000


In [4]:
large_df = df.selectExpr(
    "id",
    "repeat(cast(id as string), 1000) as large_string"
)

print("Created large dataframe")

Created large dataframe


In [5]:
spark

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 43990)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

In [ ]:
#large_df.groupBy("id").count().collect()

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OOM-Reproduction") \
    .master("local[*]") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Driver memory: 1g
Master: local[*]
Default parallelism: 12


In [4]:
df = spark.range(0, 10_000_000)

print("Records:", df.count())
print("Partitions:", df.rdd.getNumPartitions())

Records: 10000000
Partitions: 12


In [5]:
large_df = df.selectExpr(
    "id",
    "repeat(cast(id as string), 1000) as large_string"
)

print("DataFrame created")

DataFrame created


In [6]:
large_df.count()

10000000

In [7]:
test_df = large_df.limit(1_000_000)

rows = test_df.collect()

print("Rows collected:", len(rows))

Py4JJavaError: An error occurred while calling o41.collectToPython.
: java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.sql.execution.SparkPlan$$anon$1._next(SparkPlan.scala:415)
	at org.apache.spark.sql.execution.SparkPlan$$anon$1.getNext(SparkPlan.scala:426)
	at org.apache.spark.sql.execution.SparkPlan$$anon$1.getNext(SparkPlan.scala:412)
	at org.apache.spark.util.NextIterator.hasNext(NextIterator.scala:73)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach$(Iterator.scala:943)
	at org.apache.spark.util.NextIterator.foreach(NextIterator.scala:21)
	at scala.collection.generic.Growable.$plus$plus$eq(Growable.scala:62)
	at scala.collection.generic.Growable.$plus$plus$eq$(Growable.scala:53)
	at scala.collection.mutable.ArrayBuffer.$plus$plus$eq(ArrayBuffer.scala:105)
	at scala.collection.mutable.ArrayBuffer.$plus$plus$eq(ArrayBuffer.scala:49)
	at scala.collection.TraversableOnce.to(TraversableOnce.scala:366)
	at scala.collection

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Executor-OOM-Test") \
    .master("local-cluster[2,1,1024]") \
    .config("spark.executor.memory", "512m") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

Spark version: 3.5.0
Master: local-cluster[2,1,1024]


In [2]:
print("SparkContext:", spark.sparkContext)

SparkContext: <SparkContext master=local-cluster[2,1,1024] appName=Executor-OOM-Test>


In [3]:
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Master: local-cluster[2,1,1024]
Default parallelism: 2


In [4]:
from pyspark.sql import functions as F

df = spark.range(0, 2_000_000, numPartitions=2)

big_df = df.select(
    "id",
    F.repeat(F.col("id").cast("string"), 5000).alias("large_string")
)

print("Partitions:", big_df.rdd.getNumPartitions())

Partitions: 2


In [5]:
big_df.count()

2000000

In [6]:

big_df = spark.range(
    0,
    10_000_000,
    numPartitions=2
).select(
    "id",
    F.repeat(F.col("id").cast("string"), 10000).alias("large_string")
)

print("Partitions:", big_df.rdd.getNumPartitions())

Partitions: 2


In [7]:
big_df.count()

10000000

In [8]:
from pyspark.sql import functions as F

df = spark.range(0, 5_000_000, numPartitions=2)

heavy_df = df.groupBy(
    (F.col("id") % 2).alias("group_id")
).agg(
    F.collect_list(
        F.repeat(F.col("id").cast("string"), 1000)
    ).alias("all_values")
)

print("Starting aggregation...")
heavy_df.count()

Starting aggregation...


2

In [9]:
result = heavy_df.rdd.map(
    lambda row: len(row.all_values)
).collect()

print(result)

Py4JJavaError: An error occurred while calling o67.javaToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 12.0 failed 4 times, most recent failure: Lost task 1.8 in stage 12.0 (TID 21) (172.17.0.2 executor 6): java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.unsafe.types.UTF8String.repeat(UTF8String.java:735)
	at org.apache.spark.sql.catalyst.expressions.StringRepeat.nullSafeEval(stringExpressions.scala:1831)
	at org.apache.spark.sql.catalyst.expressions.BinaryExpression.eval(Expression.scala:672)
	at org.apache.spark.sql.catalyst.expressions.aggregate.Collect.update(collect.scala:53)
	at org.apache.spark.sql.catalyst.expressions.aggregate.Collect.update(collect.scala:39)
	at org.apache.spark.sql.catalyst.expressions.aggregate.TypedImperativeAggregate.update(interfaces.scala:583)
	at org.apache.spark.sql.execution.aggregate.AggregationIterator$$anonfun$1.$anonfun$applyOrElse$2(AggregationIterator.scala:197)
	at org.apache.spar

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

df = spark.range(0, 10_000_000, numPartitions=2)

shuffle_df = (
    df.withColumn("key", F.col("id") % 2)
      .groupBy("key")
      .count()
)

shuffle_df.explain(True)

== Parsed Logical Plan ==
'Aggregate ['key], ['key, count(1) AS count#8L]
+- Project [id#0L, (id#0L % cast(2 as bigint)) AS key#2L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Analyzed Logical Plan ==
key: bigint, count: bigint
Aggregate [key#2L], [key#2L, count(1) AS count#8L]
+- Project [id#0L, (id#0L % cast(2 as bigint)) AS key#2L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Optimized Logical Plan ==
Aggregate [key#2L], [key#2L, count(1) AS count#8L]
+- Project [(id#0L % 2) AS key#2L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[key#2L], functions=[count(1)], output=[key#2L, count#8L])
   +- Exchange hashpartitioning(key#2L, 200), ENSURE_REQUIREMENTS, [plan_id=15]
      +- HashAggregate(keys=[key#2L], functions=[partial_count(1)], output=[key#2L, count#12L])
         +- Project [(id#0L % 2) AS key#2L]
            +- Range (0, 10000000, step=1, splits=2)



In [2]:
result = shuffle_df.collect()

print(result)

[Row(key=0, count=5000000), Row(key=1, count=5000000)]


In [3]:
shuffle_df.rdd.getNumPartitions()

1

In [4]:
print(spark.conf.get("spark.sql.adaptive.enabled"))

true


In [5]:
from pyspark.sql import functions as F

skew_df = spark.range(0, 10_000_000, numPartitions=2) \
    .withColumn(
        "key",
        F.when(F.col("id") < 9_900_000, 0).otherwise(F.col("id"))
    )

skew_df.groupBy("key").count().show(10, False)

+-------+-------+
|key    |count  |
+-------+-------+
|0      |9900000|
|9900089|1      |
|9900250|1      |
|9900882|1      |
|9901155|1      |
|9901319|1      |
|9901843|1      |
|9902006|1      |
|9902114|1      |
|9902394|1      |
+-------+-------+
only showing top 10 rows



In [6]:
skew_result = skew_df.groupBy("key").count()
skew_result.explain(True)

== Parsed Logical Plan ==
'Aggregate ['key], ['key, count(1) AS count#38L]
+- Project [id#13L, CASE WHEN (id#13L < cast(9900000 as bigint)) THEN cast(0 as bigint) ELSE id#13L END AS key#15L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Analyzed Logical Plan ==
key: bigint, count: bigint
Aggregate [key#15L], [key#15L, count(1) AS count#38L]
+- Project [id#13L, CASE WHEN (id#13L < cast(9900000 as bigint)) THEN cast(0 as bigint) ELSE id#13L END AS key#15L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Optimized Logical Plan ==
Aggregate [key#15L], [key#15L, count(1) AS count#38L]
+- Project [CASE WHEN (id#13L < 9900000) THEN 0 ELSE id#13L END AS key#15L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[key#15L], functions=[count(1)], output=[key#15L, count#38L])
   +- Exchange hashpartitioning(key#15L, 200), ENSURE_REQUIREMENTS, [plan_id=98]
      +- HashAggregate(keys=[key#15L], functions=[p

In [7]:
from pyspark.sql import functions as F

skew_df.groupBy("key") \
       .count() \
       .orderBy(F.desc("count")) \
       .show(10, False)

+-------+-------+
|key    |count  |
+-------+-------+
|0      |9900000|
|9900089|1      |
|9900250|1      |
|9900882|1      |
|9901155|1      |
|9901319|1      |
|9901843|1      |
|9902006|1      |
|9902114|1      |
|9902394|1      |
+-------+-------+
only showing top 10 rows



In [8]:
partition_sizes = (
    skew_df.rdd
    .mapPartitions(lambda x: [sum(1 for _ in x)])
    .collect()
)


print(partition_sizes)

[5000000, 5000000]


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 33380)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Shuffle-Failure-Test") \
    .master("local-cluster[2,1,1024]") \
    .config("spark.executor.memory", "512m") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

print(spark.version)

3.5.0


In [2]:
from pyspark.sql import functions as F

df = spark.range(0, 20_000_000, numPartitions=2)

shuffle_df = (
    df.withColumn("key", (F.rand() * 100000).cast("int"))
      .repartition(2, "key")
)

In [3]:
shuffle_df.count()

20000000

In [4]:
shuffle_df.explain(True)

== Parsed Logical Plan ==
'RepartitionByExpression ['key], 2
+- Project [id#0L, cast((rand(-1452189374826180853) * cast(100000 as double)) as int) AS key#2]
   +- Range (0, 20000000, step=1, splits=Some(2))

== Analyzed Logical Plan ==
id: bigint, key: int
RepartitionByExpression [key#2], 2
+- Project [id#0L, cast((rand(-1452189374826180853) * cast(100000 as double)) as int) AS key#2]
   +- Range (0, 20000000, step=1, splits=Some(2))

== Optimized Logical Plan ==
RepartitionByExpression [key#2], 2
+- Project [id#0L, cast((rand(-1452189374826180853) * 100000.0) as int) AS key#2]
   +- Range (0, 20000000, step=1, splits=Some(2))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange hashpartitioning(key#2, 2), REPARTITION_BY_NUM, [plan_id=86]
   +- Project [id#0L, cast((rand(-1452189374826180853) * 100000.0) as int) AS key#2]
      +- Range (0, 20000000, step=1, splits=2)



In [6]:
spark.sparkContext.uiWebUrl

'http://5c9c8c4f9ccb:4040'

In [7]:
from pyspark.sql import functions as F

df1 = spark.range(0, 5_000_000, numPartitions=2) \
    .withColumn("key", F.col("id") % 100)

df2 = spark.range(0, 5_000_000, numPartitions=2) \
    .withColumn("key", F.col("id") % 100)

joined = df1.join(df2, "key")

joined.count()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [10]:
from pyspark.sql import functions as F

df1 = spark.range(0, 2_000_000, numPartitions=2) \
    .withColumn("key", F.col("id") % 1000)

df2 = spark.range(0, 2_000_000, numPartitions=2) \
    .withColumn("key", F.col("id") % 1000)

print("df1:", df1.count())
print("df2:", df2.count())

df1: 2000000
df2: 2000000


In [11]:
joined = df1.join(df2, "key")

print("Join created. Now executing...")
print("Partitions:", joined.rdd.getNumPartitions())

Join created. Now executing...
Partitions: 2


In [12]:
result = joined.groupBy("key").count()

print("Shuffle operation created.")
result.explain()

Shuffle operation created.
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[key#35L], functions=[count(1)])
   +- HashAggregate(keys=[key#35L], functions=[partial_count(1)])
      +- Project [key#35L]
         +- SortMergeJoin [key#35L], [key#40L], Inner
            :- Sort [key#35L ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(key#35L, 2), ENSURE_REQUIREMENTS, [plan_id=453]
            :     +- Project [(id#33L % 1000) AS key#35L]
            :        +- Filter isnotnull((id#33L % 1000))
            :           +- Range (0, 2000000, step=1, splits=2)
            +- Sort [key#40L ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(key#40L, 2), ENSURE_REQUIREMENTS, [plan_id=454]
                  +- Project [(id#38L % 1000) AS key#40L]
                     +- Filter isnotnull((id#38L % 1000))
                        +- Range (0, 2000000, step=1, splits=2)




In [18]:
print("Executing shuffle...")
result.count()
print("Shuffle completed successfully")

Executing shuffle...
Shuffle completed successfully


In [15]:
print(spark.sparkContext.getConf().get("spark.executor.instances", "not set"))
print(spark.sparkContext.master)

not set
local-cluster[2,1,1024]


In [19]:
from pyspark.sql import functions as F
import time

failure_df = spark.range(
    0, 20_000_000,
    numPartitions=2
).withColumn(
    "key",
    (F.col("id") % 100000)
)

failure_result = failure_df.groupBy("key").count()

print("Starting shuffle...")
failure_result.count()

print("Shuffle completed")

Starting shuffle...
Shuffle completed
